### Homework 5: Question search engine

Remeber week01 where you used GloVe embeddings to find related questions? That was.. cute, but far from state of the art. It's time to really solve this task using context-aware embeddings.

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [1]:
#%pip install --upgrade datasets 
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

c:\Users\salni\CHATTER\chatterVenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


### Load data and model

In [3]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [4]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

### Tokenize the data

In [5]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [6]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Task 1: evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [6]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [7]:
for batch in val_loader:
     break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
  predicted = model(
      input_ids=batch['input_ids'],
      attention_mask=batch['attention_mask'],
      token_type_ids=batch['token_type_ids']
  )

print('\nPrediction (probs):', torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

__Your task__ is to measure the validation accuracy of your model.
Doing so naively may take several hours. Please make sure you use the following optimizations:

- run the model on GPU with no_grad
- using batch size larger than 1
- use optimize data loader with num_workers > 1
- (optional) use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [9]:

model = model.to(device)

In [10]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score

val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=2
)

true_values = []
pred_values = []
for batch in tqdm(val_loader):
    # here be your training code
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
      predicted = model(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        token_type_ids=batch['token_type_ids'])

    pred = torch.argmax(predicted.logits, dim=1).data

    true_values += list(batch['labels'].data.cpu().numpy())
    pred_values += list(pred.cpu().numpy())
print("Sample batch:", batch)

accuracy = accuracy_score(true_values, pred_values)


100%|██████████| 2527/2527 [02:35<00:00, 16.23it/s]

Sample batch: {'labels': tensor([0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0'), 'idx': tensor([40416, 40417, 40418, 40419, 40420, 40421, 40422, 40423, 40424, 40425,
        40426, 40427, 40428, 40429], device='cuda:0'), 'input_ids': tensor([[ 101, 1327, 1110,  ...,    0,    0,    0],
        [ 101, 1327, 1132,  ...,    0,    0,    0],
        [ 101, 1327, 1110,  ...,    0,    0,    0],
        ...,
        [ 101, 2181, 2903,  ...,    0,    0,    0],
        [ 101, 1731, 1202,  ...,    0,    0,    0],
        [ 101, 1731, 1169,  ...,    0,    0,    0]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ...

In [11]:
assert 0.9 < accuracy < 0.91

### Task 2: train the model (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

In [8]:
from peft import LoraConfig, get_peft_model, TaskType

In [9]:
# Load the model and tokenizer
model_name = "microsoft/deberta-v3-small"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
#model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained('deberta-v3-lora-qqp')
model = model.to(device)

c:\Users\salni\CHATTER\chatterVenv\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of the model checkpoint at deberta-v3-lora-qqp were not used when initializing DebertaV2ForSequenceClassification: ['classifier.modules_to_save.default.bias', 'classifier.modules_to_save.default.weight', 'classifier.original_module.bias', 'classifier.original_module.weight', 'deberta.encoder.layer.0.attention.self.query_proj.base_layer.bias', 'deberta.encoder.layer.0.attention.self.query_proj.base_layer.weight', 'deberta.encoder.layer.0.attention.self.query_proj.lora_A.de

In [10]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

# Подготовка данных для обучения
train_set = qqp_preprocessed['train']
train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=16, shuffle=True, collate_fn=transformers.default_data_collator,
    num_workers=2
)

# Валидационный загрузчик (оставляем как было)
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=2
)

Map: 100%|██████████| 40430/40430 [00:02<00:00, 13499.91 examples/s]


In [11]:
# Настройка LoRA для DeBERTa-v3
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,  # rank
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query_proj", "value_proj"]  # Для DeBERTa-v3
)

# Применяем LoRA к модели
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

# Настройка аргументов обучения
training_args = transformers.TrainingArguments(
    output_dir="./results",
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",  # или "tensorboard", "wandb"
    remove_unused_columns=False,  # Важно для PEFT!
    fp16=True,  # Включить если используете GPU с поддержкой FP16
    dataloader_pin_memory=False,
)

# Создание Trainer и запуск обучения
trainer = transformers.Trainer(
     model=lora_model,
     args=training_args,
     train_dataset= qqp_preprocessed['train'],
     eval_dataset= qqp_preprocessed['validation'],
)

# Оптимизатор только для обучаемых параметров
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

trainable params: 148,994 || all params: 142,045,444 || trainable%: 0.1049


No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [14]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# Цикл обучения
num_epochs = 10
for epoch in range(num_epochs):
    # Обучение
    lora_model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        batch_on_dev = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = lora_model(
            input_ids=batch_on_dev['input_ids'],
            attention_mask=batch_on_dev['attention_mask'],
            token_type_ids=batch_on_dev['token_type_ids'],
            labels=batch_on_dev['labels']
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Валидация
    lora_model.eval()
    true_values = []
    pred_values = []
    val_loss = 0

    for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.no_grad():
            outputs = lora_model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                token_type_ids=batch['token_type_ids'],
                labels=batch['labels']
            )

            predicted = outputs.logits
            pred = torch.argmax(predicted, dim=1).data

            true_values += list(batch['labels'].data.cpu().numpy())
            pred_values += list(pred.cpu().numpy())
            val_loss += outputs.loss.item()

    # Вычисление метрик
    train_loss_avg = train_loss / len(train_loader)
    val_loss_avg = val_loss / len(val_loader)
    accuracy = accuracy_score(true_values, pred_values)
    

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss_avg:.4f}")
    print(f"Val Loss: {val_loss_avg:.4f}")
    print(f"Val Accuracy: {accuracy:.4f}")
    print("-" * 50)
    if accuracy >= 0.9:
        break

Epoch 1 Validation: 100%|██████████| 2527/2527 [02:09<00:00, 19.57it/s]


Epoch 1/10
Train Loss: 0.3186
Val Loss: 0.2817
Val Accuracy: 0.8779
--------------------------------------------------


Epoch 2 Validation: 100%|██████████| 2527/2527 [02:03<00:00, 20.39it/s]


Epoch 2/10
Train Loss: 0.2700
Val Loss: 0.2561
Val Accuracy: 0.8890
--------------------------------------------------


Epoch 3 Validation: 100%|██████████| 2527/2527 [02:24<00:00, 17.46it/s]


...

Epoch 4/10
Train Loss: 0.2397
Val Loss: 0.2439
Val Accuracy: 0.8983
--------------------------------------------------


Epoch 5 Validation: 100%|██████████| 2527/2527 [03:21<00:00, 12.54it/s]


Epoch 5/10
Train Loss: 0.2312
Val Loss: 0.2421
Val Accuracy: 0.8979
--------------------------------------------------


Epoch 6 Validation: 100%|██████████| 2527/2527 [01:50<00:00, 22.79it/s]

Epoch 6/10
Train Loss: 0.2241
Val Loss: 0.2385
Val Accuracy: 0.9001
--------------------------------------------------


In [15]:
# Сохранение модели LoRA
lora_model.save_pretrained("deberta-v3-small-lora-qqp")

In [16]:
# Финальная валидация после обучения
lora_model.eval()
true_values = []
pred_values = []

for batch in tqdm(val_loader, desc="Final Validation"):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = lora_model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            token_type_ids=batch['token_type_ids']
        )

        pred = torch.argmax(outputs.logits, dim=1).data
        true_values += list(batch['labels'].data.cpu().numpy())
        pred_values += list(pred.cpu().numpy())

final_accuracy = accuracy_score(true_values, pred_values)
print(f"Final Validation Accuracy: {final_accuracy:.4f}")

# Сохранение модели LoRA
model.save_pretrained("deberta-v3-lora-qqp")

Final Validation: 100%|██████████| 2527/2527 [01:48<00:00, 23.27it/s]


Final Validation Accuracy: 0.9001


### Task 3: try the full pipeline (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

In [40]:
# Load your pre-trained model and tokenizer
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [8]:
from tqdm import tqdm

In [9]:
class DuplicateFinder:
  def __init__(self, model_name="gchhablani/bert-base-cased-finetuned-qqp"):
    self.tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    self.model = transformers.AutoModelForSequenceClassification.from_pretrained("gchhablani/bert-base-cased-finetuned-qqp")
    self.model.to(device)

    self.model.eval()

    self.train_questions = [qqp['train'][i]['text1'] for i in range(min(3000, len(qqp['train'])))]

  def find_duplicates(self, query_question, top_k=5):
    similarities = []

    for train_question in tqdm(self.train_questions):
      if query_question == train_question:
        continue
      
      
      inputs = self.tokenizer(
              query_question, train_question,
              padding='max_length', max_length=MAX_LENGTH, truncation=True)
      inputs['label'] = train_question['label']
      inputs = {k: v.to(device) for k, v in inputs.items()}
      predicted = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            token_type_ids=inputs['token_type_ids'])
      preds = torch.softmax(predicted.logits, dim=1)
      prob = preds[0, 1].item()

      similarities += [(train_question, prob)]

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

In [10]:
len(qqp['train'][0]['text1'])

75

In [11]:
finder = DuplicateFinder()
querys = ['Where is the library?',
          'How fast were you going?',
          'What time is it?',
          'How to basic',
          'LaLeLuLeLo']

for q in querys:
  print(f"Query:{q}")
  print(finder.find_duplicates(q))

Query:Where is the library?


  0%|          | 0/3000 [00:00<?, ?it/s]


TypeError: string indices must be integers, not 'str'

In [12]:
import datasets
from transformers import AutoTokenizer, AutoModel
import torch

# 1. Load the QQP dataset
dataset = datasets.load_dataset('glue', 'qqp')
train_questions = dataset['train']

# 2. Load your model's tokenizer and the base BERT model for embeddings
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the base model, not the classification model, to get hidden states
embedding_model = AutoModel.from_pretrained(model_name)
embedding_model.to(device)  # Your device (e.g., 'cuda' or 'cpu')
embedding_model.eval()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating test split: 100%|██████████| 390965/390965 [00:00<00:00, 2595096.75 examples/s]


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [14]:
def get_bert_embedding(text, model, tokenizer, device):
    """Generate a sentence embedding for a single text."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        # Use mean pooling over the last hidden state (excluding padding tokens)
        # Shape: (batch_size, seq_len, hidden_size) -> (batch_size, hidden_size)
        last_hidden_state = outputs.last_hidden_state
        attention_mask = inputs['attention_mask']
        
        # Expand mask to match hidden state dimensions and compute mean
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
    
    return mean_embeddings.cpu().squeeze()

# Generate and store embeddings for all training questions
# This might take a while. Consider processing in batches.
train_embeddings = []
train_texts = []  # Store the original text for retrieval

batch_size = 32
for i in tqdm(range(0, len(train_questions), batch_size)):
    batch_texts = train_questions[i:i+batch_size]['question1'] + train_questions[i:i+batch_size]['question2']
    for text in batch_texts:
        if text:  # Handle any potential None values
            emb = get_bert_embedding(text, embedding_model, tokenizer, device)
            train_embeddings.append(emb)
            train_texts.append(text)

# Convert list of embeddings to a tensor for efficient computation
train_embedding_matrix = torch.stack(train_embeddings)

100%|██████████| 11371/11371 [1:40:56<00:00,  1.88it/s]


In [15]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Your 5 example questions
example_questions = [
    "Why in India do we not have one on one political debate as in USA?",
    "What is OnePlus One?",
    "Does our mind control our emotions?",
    "What made Islam spread around the world faster than any other religion?",
    "What do people around the world typically wear to sleep?"
]

top_k = 5
results = {}

for query in example_questions:
    # 1. Get embedding for the example question
    query_embedding = get_bert_embedding(query, embedding_model, tokenizer, device).numpy().reshape(1, -1)
    
    # 2. Calculate cosine similarity with all training embeddings
    # Convert the training embedding matrix to numpy for use with scikit-learn
    train_embedding_np = train_embedding_matrix.numpy()
    cos_similarities = cosine_similarity(query_embedding, train_embedding_np)[0]
    
    # 3. Get indices of top K most similar questions
    top_indices = np.argsort(cos_similarities)[-top_k:][::-1]
    
    # 4. Store the results
    top_matches = []
    for idx in top_indices:
        top_matches.append({
            'question': train_texts[idx],
            'similarity_score': cos_similarities[idx]
        })
    results[query] = top_matches

In [23]:
for k, v in results.items():
    print('Initial question:', k)
    print(30 * '-')
    for dupli in v:
        isdupli = ''
        if dupli['similarity_score'] >= 1.0:
            isdupli = '-DUPLICATE'
        print(str(dupli) + isdupli)
    print(30 * '-')

Initial question: Why in India do we not have one on one political debate as in USA?
------------------------------
{'question': 'Why in India do we not have one on one political debate as in USA?', 'similarity_score': np.float32(1.0000001)}-DUPLICATE
{'question': 'Why in India do we not have one on one political debate as in USA?', 'similarity_score': np.float32(1.0000001)}-DUPLICATE
{'question': 'Why in India do we not have one on one political debate as in USA?', 'similarity_score': np.float32(1.0000001)}-DUPLICATE
{'question': 'Why are many indian girls liberal about pre marital sex?', 'similarity_score': np.float32(0.9355529)}
{'question': 'Why my downvote history is not shown in my activity?', 'similarity_score': np.float32(0.9330742)}
------------------------------
Initial question: What is OnePlus One?
------------------------------
{'question': 'What is OnePlus One?', 'similarity_score': np.float32(0.9999998)}
{'question': 'What is OnePlus?', 'similarity_score': np.float32(0.9